In [2]:
import sys

!{sys.executable} -m pip install "greenlet==3.1.1" --only-binary=:all:

You should consider upgrading via the 'c:\users\india\appdata\local\programs\python\python39\python.exe -m pip install --upgrade pip' command.


In [3]:
import sys

!{sys.executable} -m pip install sqlalchemy psycopg2-binary

  Using cached sqlalchemy-2.0.52-cp39-cp39-win_amd64.whl (2.2 MB)
  Using cached psycopg2_binary-2.9.12-cp39-cp39-win_amd64.whl (2.8 MB)


You should consider upgrading via the 'c:\users\india\appdata\local\programs\python\python39\python.exe -m pip install --upgrade pip' command.


In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from getpass import getpass

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
df = pd.read_csv("../data/processed/workforce_processed.csv")

print("Dataset shape:", df.shape)

Dataset shape: (1470, 37)


In [3]:
print("Columns:")
for i, column in enumerate(df.columns, start=1):
    print(i, column)

Columns:
1 Age
2 Attrition
3 BusinessTravel
4 DailyRate
5 Department
6 DistanceFromHome
7 Education
8 EducationField
9 EmployeeID
10 EnvironmentSatisfaction
11 Gender
12 HourlyRate
13 JobInvolvement
14 JobLevel
15 JobRole
16 JobSatisfaction
17 MaritalStatus
18 MonthlyIncome
19 MonthlyRate
20 NumCompaniesWorked
21 OverTime
22 PercentSalaryHike
23 PerformanceRating
24 RelationshipSatisfaction
25 StockOptionLevel
26 TotalWorkingYears
27 TrainingTimesLastYear
28 WorkLifeBalance
29 YearsAtCompany
30 YearsInCurrentRole
31 YearsSinceLastPromotion
32 YearsWithCurrManager
33 AttritionFlag
34 TenureGroup
35 AgeGroup
36 YearsSincePromotionRatio
37 ExperienceGroup


In [4]:
from getpass import getpass
from sqlalchemy import create_engine, text

DB_USER = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "workforce_intelligence"

DB_PASSWORD = getpass("Enter PostgreSQL password: ")

Enter PostgreSQL password:  ········


In [5]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database engine created successfully.")

Database engine created successfully.


In [7]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

print("Engine created successfully.")

Engine created successfully.


In [8]:
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("Database connection successful:", result.fetchone())

except Exception as e:
    print("Database connection failed.")
    print(type(e).__name__)
    print(str(e))

Database connection successful: (1,)


In [9]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """))

    tables = [row[0] for row in result]

print("Tables in PostgreSQL:")
for table in tables:
    print("-", table)

Tables in PostgreSQL:
- attrition
- compensation
- employees
- employment
- engagement
- performance_learning


In [10]:
employees_df = df[
    [
        "EmployeeID",
        "Age",
        "Gender",
        "MaritalStatus",
        "Education",
        "EducationField"
    ]
].copy()

In [11]:
employees_df = employees_df.rename(columns={
    "EmployeeID": "employee_id",
    "Age": "age",
    "Gender": "gender",
    "MaritalStatus": "marital_status",
    "Education": "education",
    "EducationField": "education_field"
})

In [12]:
print("Rows:", len(employees_df))
print("Unique Employee IDs:", employees_df["employee_id"].nunique())
print("Missing Employee IDs:", employees_df["employee_id"].isna().sum())

Rows: 1470
Unique Employee IDs: 1470
Missing Employee IDs: 0


In [13]:
employees_df.to_sql(
    "employees",
    engine,
    if_exists="append",
    index=False
)

print("Employees loaded successfully.")

Employees loaded successfully.


In [14]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM employees")
    )
    
    count = result.scalar()

print("Employees in PostgreSQL:", count)

Employees in PostgreSQL: 1470


In [15]:
employment_df = df[
    [
        "EmployeeID",
        "Department",
        "JobRole",
        "JobLevel",
        "BusinessTravel",
        "TotalWorkingYears",
        "YearsAtCompany",
        "YearsInCurrentRole",
        "YearsSinceLastPromotion",
        "YearsWithCurrManager",
        "NumCompaniesWorked",
        "DistanceFromHome",
        "OverTime"
    ]
].copy()

In [16]:
employment_df = employment_df.rename(columns={
    "EmployeeID": "employee_id",
    "Department": "department",
    "JobRole": "job_role",
    "JobLevel": "job_level",
    "BusinessTravel": "business_travel",
    "TotalWorkingYears": "total_working_years",
    "YearsAtCompany": "years_at_company",
    "YearsInCurrentRole": "years_in_current_role",
    "YearsSinceLastPromotion": "years_since_last_promotion",
    "YearsWithCurrManager": "years_with_curr_manager",
    "NumCompaniesWorked": "num_companies_worked",
    "DistanceFromHome": "distance_from_home",
    "OverTime": "overtime"
})

In [17]:
print("Rows:", len(employment_df))
print("Unique Employee IDs:", employment_df["employee_id"].nunique())
print("Missing Employee IDs:", employment_df["employee_id"].isna().sum())

Rows: 1470
Unique Employee IDs: 1470
Missing Employee IDs: 0


In [18]:
employment_df.isna().sum()

employee_id                   0
department                    0
job_role                      0
job_level                     0
business_travel               0
total_working_years           0
years_at_company              0
years_in_current_role         0
years_since_last_promotion    0
years_with_curr_manager       0
num_companies_worked          0
distance_from_home            0
overtime                      0
dtype: int64

In [19]:
employment_df.to_sql(
    "employment",
    engine,
    if_exists="append",
    index=False
)

print("Employment data loaded successfully.")

Employment data loaded successfully.


In [20]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM employment")
    )
    
    count = result.scalar()

print("Employment records in PostgreSQL:", count)

Employment records in PostgreSQL: 1470


In [21]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*)
        FROM employment e
        LEFT JOIN employees emp
            ON e.employee_id = emp.employee_id
        WHERE emp.employee_id IS NULL;
    """))
    
    unmatched = result.scalar()

print("Unmatched employment records:", unmatched)

Unmatched employment records: 0


In [22]:
compensation_df = df[
    [
        "EmployeeID",
        "DailyRate",
        "HourlyRate",
        "MonthlyIncome",
        "MonthlyRate",
        "PercentSalaryHike",
        "StockOptionLevel"
    ]
].copy()

In [23]:
compensation_df = compensation_df.rename(columns={
    "EmployeeID": "employee_id",
    "DailyRate": "daily_rate",
    "HourlyRate": "hourly_rate",
    "MonthlyIncome": "monthly_income",
    "MonthlyRate": "monthly_rate",
    "PercentSalaryHike": "percent_salary_hike",
    "StockOptionLevel": "stock_option_level"
})

In [24]:
print("Rows:", len(compensation_df))
print("Unique Employee IDs:", compensation_df["employee_id"].nunique())
print("Missing Employee IDs:", compensation_df["employee_id"].isna().sum())

Rows: 1470
Unique Employee IDs: 1470
Missing Employee IDs: 0


In [25]:
compensation_df.isna().sum()

employee_id            0
daily_rate             0
hourly_rate            0
monthly_income         0
monthly_rate           0
percent_salary_hike    0
stock_option_level     0
dtype: int64

In [26]:
compensation_df.to_sql(
    "compensation",
    engine,
    if_exists="append",
    index=False
)

print("Compensation data loaded successfully.")

Compensation data loaded successfully.


In [27]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM compensation")
    )
    
    count = result.scalar()

print("Compensation records in PostgreSQL:", count)

Compensation records in PostgreSQL: 1470


In [28]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*)
        FROM compensation c
        LEFT JOIN employees e
            ON c.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """))
    
    unmatched = result.scalar()

print("Unmatched compensation records:", unmatched)

Unmatched compensation records: 0


In [29]:
engagement_df = df[
    [
        "EmployeeID",
        "EnvironmentSatisfaction",
        "JobInvolvement",
        "JobSatisfaction",
        "RelationshipSatisfaction",
        "WorkLifeBalance"
    ]
].copy()

In [30]:
engagement_df = engagement_df.rename(columns={
    "EmployeeID": "employee_id",
    "EnvironmentSatisfaction": "environment_satisfaction",
    "JobInvolvement": "job_involvement",
    "JobSatisfaction": "job_satisfaction",
    "RelationshipSatisfaction": "relationship_satisfaction",
    "WorkLifeBalance": "work_life_balance"
})

In [31]:
print("Rows:", len(engagement_df))
print("Unique Employee IDs:", engagement_df["employee_id"].nunique())
print("Missing Employee IDs:", engagement_df["employee_id"].isna().sum())

Rows: 1470
Unique Employee IDs: 1470
Missing Employee IDs: 0


In [32]:
engagement_df.isna().sum()

employee_id                  0
environment_satisfaction     0
job_involvement              0
job_satisfaction             0
relationship_satisfaction    0
work_life_balance            0
dtype: int64

In [33]:
engagement_df.to_sql(
    "engagement",
    engine,
    if_exists="append",
    index=False
)

print("Engagement data loaded successfully.")

Engagement data loaded successfully.


In [34]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM engagement")
    )
    
    count = result.scalar()

print("Engagement records in PostgreSQL:", count)

Engagement records in PostgreSQL: 1470


In [35]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*)
        FROM engagement en
        LEFT JOIN employees e
            ON en.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """))
    
    unmatched = result.scalar()

print("Unmatched engagement records:", unmatched)

Unmatched engagement records: 0


In [36]:
performance_learning_df = df[
    [
        "EmployeeID",
        "PerformanceRating",
        "TrainingTimesLastYear"
    ]
].copy()

In [37]:
performance_learning_df = performance_learning_df.rename(columns={
    "EmployeeID": "employee_id",
    "PerformanceRating": "performance_rating",
    "TrainingTimesLastYear": "training_times_last_year"
})

In [38]:
print("Rows:", len(performance_learning_df))
print(
    "Unique Employee IDs:",
    performance_learning_df["employee_id"].nunique()
)
print(
    "Missing Employee IDs:",
    performance_learning_df["employee_id"].isna().sum()
)

Rows: 1470
Unique Employee IDs: 1470
Missing Employee IDs: 0


In [39]:
performance_learning_df.isna().sum()

employee_id                 0
performance_rating          0
training_times_last_year    0
dtype: int64

In [40]:
performance_learning_df.to_sql(
    "performance_learning",
    engine,
    if_exists="append",
    index=False
)

print("Performance and learning data loaded successfully.")

Performance and learning data loaded successfully.


In [41]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM performance_learning")
    )

    count = result.scalar()

print("Performance-learning records in PostgreSQL:", count)

Performance-learning records in PostgreSQL: 1470


In [42]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*)
        FROM performance_learning p
        LEFT JOIN employees e
            ON p.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """))

    unmatched = result.scalar()

print("Unmatched performance-learning records:", unmatched)

Unmatched performance-learning records: 0


In [43]:
def load_table(
    dataframe,
    table_name,
    required_columns,
    engine
):
    """
    Validate and load a DataFrame into PostgreSQL.
    """

    data = dataframe[required_columns].copy()

    # Validate employee IDs
    if data["employee_id"].isna().any():
        raise ValueError(
            f"{table_name}: Missing employee IDs found."
        )

    if data["employee_id"].duplicated().any():
        raise ValueError(
            f"{table_name}: Duplicate employee IDs found."
        )

    # Load into PostgreSQL
    data.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False
    )

    # Verify database count
    with engine.connect() as connection:
        result = connection.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        )

        count = result.scalar()

    print(f"{table_name} loaded successfully.")
    print(f"Records in PostgreSQL: {count}")

    return data

In [44]:
performance_learning_df = df.rename(columns={
    "EmployeeID": "employee_id",
    "PerformanceRating": "performance_rating",
    "TrainingTimesLastYear": "training_times_last_year"
})

In [45]:
performance_learning_df = load_table(
    performance_learning_df,
    "performance_learning",
    [
        "employee_id",
        "performance_rating",
        "training_times_last_year"
    ],
    engine
)

IntegrityError: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "performance_learning_pkey"
DETAIL:  Key (employee_id)=(1) already exists.

[SQL: INSERT INTO performance_learning (employee_id, performance_rating, training_times_last_year) VALUES (%(employee_id__0)s, %(performance_rating__0)s, %(training_times_last_year__0)s), (%(employee_id__1)s, %(performance_rating__1)s, %(training_times_las ... 87418 characters truncated ... year__998)s), (%(employee_id__999)s, %(performance_rating__999)s, %(training_times_last_year__999)s)]
[parameters: {'performance_rating__0': 3, 'training_times_last_year__0': 0, 'employee_id__0': 1, 'performance_rating__1': 4, 'training_times_last_year__1': 3, 'employee_id__1': 2, 'performance_rating__2': 3, 'training_times_last_year__2': 3, 'employee_id__2': 4, 'performance_rating__3': 3, 'training_times_last_year__3': 3, 'employee_id__3': 5, 'performance_rating__4': 3, 'training_times_last_year__4': 3, 'employee_id__4': 7, 'per

In [46]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM performance_learning")
    )
    count = result.scalar()

print("Current performance_learning records:", count)

Current performance_learning records: 1470


In [47]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*)
        FROM performance_learning p
        LEFT JOIN employees e
            ON p.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """))

    unmatched = result.scalar()

print("Unmatched records:", unmatched)

Unmatched records: 0


In [48]:
with engine.begin() as connection:
    connection.execute(
        text("TRUNCATE TABLE performance_learning")
    )

print("performance_learning table cleared.")

performance_learning table cleared.


In [49]:
performance_learning_df = load_table(
    performance_learning_df,
    "performance_learning",
    [
        "employee_id",
        "performance_rating",
        "training_times_last_year"
    ],
    engine
)

performance_learning loaded successfully.
Records in PostgreSQL: 1470


In [50]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM performance_learning")
    )
    count = result.scalar()

print("Current performance_learning records:", count)

Current performance_learning records: 1470


In [51]:
tables = [
    "employees",
    "employment",
    "compensation",
    "engagement",
    "performance_learning",
    "attrition"
]

with engine.connect() as connection:

    for table in tables:

        result = connection.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )

        count = result.scalar()

        print(f"{table:25} : {count} records")

employees                 : 1470 records
employment                : 1470 records
compensation              : 1470 records
engagement                : 1470 records
performance_learning      : 1470 records
attrition                 : 0 records


In [52]:
validation_query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT employee_id) AS unique_employee_ids
FROM employees;
"""

with engine.connect() as connection:
    result = connection.execute(text(validation_query))
    row = result.fetchone()

print("Total records:", row[0])
print("Unique employee IDs:", row[1])

Total records: 1470
Unique employee IDs: 1470


In [53]:
relationship_checks = {
    "employment": """
        SELECT COUNT(*)
        FROM employment em
        LEFT JOIN employees e
            ON em.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """,

    "compensation": """
        SELECT COUNT(*)
        FROM compensation c
        LEFT JOIN employees e
            ON c.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """,

    "engagement": """
        SELECT COUNT(*)
        FROM engagement en
        LEFT JOIN employees e
            ON en.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """,

    "performance_learning": """
        SELECT COUNT(*)
        FROM performance_learning p
        LEFT JOIN employees e
            ON p.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """,

    "attrition": """
        SELECT COUNT(*)
        FROM attrition a
        LEFT JOIN employees e
            ON a.employee_id = e.employee_id
        WHERE e.employee_id IS NULL;
    """
}

with engine.connect() as connection:

    for table, query in relationship_checks.items():

        result = connection.execute(text(query))
        unmatched = result.scalar()

        print(f"{table:25} : {unmatched} unmatched records")

employment                : 0 unmatched records
compensation              : 0 unmatched records
engagement                : 0 unmatched records
performance_learning      : 0 unmatched records
attrition                 : 0 unmatched records


employees
     +
employment
     +
compensation
     +
engagement
     +
performance_learning
     +
attrition
          ↓
   workforce_analytics

In [54]:
create_view_query = """
CREATE OR REPLACE VIEW workforce_analytics AS
SELECT
    e.employee_id,
    e.age,
    e.gender,
    e.marital_status,
    e.education,
    e.education_field,

    em.job_level,
    em.job_role,
    em.business_travel,
    em.total_working_years,
    em.years_at_company,
    em.years_in_current_role,
    em.years_since_last_promotion,
    em.years_with_curr_manager,
    em.num_companies_worked,
    em.distance_from_home,
    em.overtime,

    c.daily_rate,
    c.hourly_rate,
    c.monthly_income,
    c.monthly_rate,
    c.percent_salary_hike,
    c.stock_option_level,

    en.environment_satisfaction,
    en.job_involvement,
    en.job_satisfaction,
    en.relationship_satisfaction,
    en.work_life_balance,

    p.performance_rating,
    p.training_times_last_year,

    a.attrition,
    a.attrition_flag

FROM employees e

LEFT JOIN employment em
    ON e.employee_id = em.employee_id

LEFT JOIN compensation c
    ON e.employee_id = c.employee_id

LEFT JOIN engagement en
    ON e.employee_id = en.employee_id

LEFT JOIN performance_learning p
    ON e.employee_id = p.employee_id

LEFT JOIN attrition a
    ON e.employee_id = a.employee_id;
"""

with engine.begin() as connection:
    connection.execute(text(create_view_query))

print("workforce_analytics view created successfully.")

workforce_analytics view created successfully.


In [55]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM workforce_analytics
        """)
    )

    count = result.scalar()

print("Records in workforce_analytics:", count)

Records in workforce_analytics: 1470


In [56]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT *
            FROM workforce_analytics
            LIMIT 10
        """)
    )

    rows = result.fetchall()

for row in rows:
    print(row)

(1, 41, 'Female', 'Single', 2, 'Life Sciences', 2, 'Sales Executive', 'Travel_Rarely', 8, 6, 4, 0, 5, 8, 1, 'Yes', 1102, 94, 5993, 19479, 11, 0, 2, 3, 4, 1, 1, 3, 0, None, None)
(2, 49, 'Male', 'Married', 1, 'Life Sciences', 2, 'Research Scientist', 'Travel_Frequently', 10, 10, 7, 1, 7, 1, 8, 'No', 279, 61, 5130, 24907, 23, 1, 3, 2, 2, 4, 3, 4, 3, None, None)
(4, 37, 'Male', 'Single', 2, 'Other', 1, 'Laboratory Technician', 'Travel_Rarely', 7, 0, 0, 0, 0, 6, 2, 'Yes', 1373, 92, 2090, 2396, 15, 0, 4, 2, 3, 2, 3, 3, 3, None, None)
(5, 33, 'Female', 'Married', 4, 'Life Sciences', 1, 'Research Scientist', 'Travel_Frequently', 8, 8, 7, 3, 0, 1, 3, 'Yes', 1392, 56, 2909, 23159, 11, 0, 4, 3, 3, 3, 3, 3, 3, None, None)
(7, 27, 'Male', 'Married', 1, 'Medical', 1, 'Laboratory Technician', 'Travel_Rarely', 6, 2, 2, 2, 2, 9, 2, 'No', 591, 40, 3468, 16632, 12, 1, 1, 3, 2, 4, 3, 3, 3, None, None)
(8, 32, 'Male', 'Single', 2, 'Life Sciences', 1, 'Laboratory Technician', 'Travel_Frequently', 8, 7, 7, 

In [58]:
with engine.begin() as connection:
    connection.execute(
        text("""
            ALTER TABLE employees
            ADD COLUMN IF NOT EXISTS department VARCHAR(100);
        """)
    )

print("Department column added to employees table.")

Department column added to employees table.


In [59]:
department_df = df[["EmployeeID", "Department"]].copy()

department_df = department_df.rename(
    columns={
        "EmployeeID": "employee_id",
        "Department": "department"
    }
)

print(department_df.head())
print("Department records:", len(department_df))
print("Unique employees:", department_df["employee_id"].nunique())

   employee_id              department
0            1                   Sales
1            2  Research & Development
2            4  Research & Development
3            5  Research & Development
4            7  Research & Development
Department records: 1470
Unique employees: 1470


In [60]:
department_records = department_df.to_dict("records")

with engine.begin() as connection:

    connection.execute(
        text("""
            UPDATE employees
            SET department = :department
            WHERE employee_id = :employee_id
        """),
        department_records
    )

print("Department data updated successfully.")

Department data updated successfully.


In [61]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS total,
                COUNT(department) AS department_available,
                COUNT(DISTINCT department) AS unique_departments
            FROM employees;
        """)
    )

    row = result.fetchone()

print("Total employees:", row[0])
print("Employees with department:", row[1])
print("Unique departments:", row[2])

Total employees: 1470
Employees with department: 1470
Unique departments: 3


In [62]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT
                department,
                COUNT(*) AS employee_count
            FROM employees
            GROUP BY department
            ORDER BY employee_count DESC;
        """)
    )

    rows = result.fetchall()

for row in rows:
    print(row)

('Research & Development', 961)
('Sales', 446)
('Human Resources', 63)


In [63]:
create_view_query = """
CREATE OR REPLACE VIEW workforce_analytics AS

SELECT
    e.employee_id,
    e.age,
    e.gender,
    e.marital_status,
    e.education,
    e.education_field,
    e.department,

    em.job_level,
    em.job_role,
    em.business_travel,
    em.total_working_years,
    em.years_at_company,
    em.years_in_current_role,
    em.years_since_last_promotion,
    em.years_with_curr_manager,
    em.num_companies_worked,
    em.distance_from_home,
    em.overtime,

    c.daily_rate,
    c.hourly_rate,
    c.monthly_income,
    c.monthly_rate,
    c.percent_salary_hike,
    c.stock_option_level,

    en.environment_satisfaction,
    en.job_involvement,
    en.job_satisfaction,
    en.relationship_satisfaction,
    en.work_life_balance,

    p.performance_rating,
    p.training_times_last_year,

    a.attrition,
    a.attrition_flag

FROM employees e

LEFT JOIN employment em
    ON e.employee_id = em.employee_id

LEFT JOIN compensation c
    ON e.employee_id = c.employee_id

LEFT JOIN engagement en
    ON e.employee_id = en.employee_id

LEFT JOIN performance_learning p
    ON e.employee_id = p.employee_id

LEFT JOIN attrition a
    ON e.employee_id = a.employee_id;
"""

with engine.begin() as connection:
    connection.execute(text(create_view_query))

print("workforce_analytics view created successfully.")

ProgrammingError: (psycopg2.errors.InvalidTableDefinition) cannot change name of view column "job_level" to "department"
HINT:  Use ALTER VIEW ... RENAME COLUMN ... to change name of view column instead.

[SQL: 
CREATE OR REPLACE VIEW workforce_analytics AS

SELECT
    e.employee_id,
    e.age,
    e.gender,
    e.marital_status,
    e.education,
    e.education_field,
    e.department,

    em.job_level,
    em.job_role,
    em.business_travel,
    em.total_working_years,
    em.years_at_company,
    em.years_in_current_role,
    em.years_since_last_promotion,
    em.years_with_curr_manager,
    em.num_companies_worked,
    em.distance_from_home,
    em.overtime,

    c.daily_rate,
    c.hourly_rate,
    c.monthly_income,
    c.monthly_rate,
    c.percent_salary_hike,
    c.stock_option_level,

    en.environment_satisfaction,
    en.job_involvement,
    en.job_satisfaction,
    en.relationship_satisfaction,
    en.work_life_balance,

    p.performance_rating,
    p.training_times_las

In [64]:
with engine.begin() as connection:
    connection.execute(
        text("DROP VIEW IF EXISTS workforce_analytics")
    )

print("Old workforce_analytics view removed.")

Old workforce_analytics view removed.


In [65]:
create_view_query = """
CREATE VIEW workforce_analytics AS

SELECT
    e.employee_id,
    e.age,
    e.gender,
    e.marital_status,
    e.education,
    e.education_field,
    e.department,

    em.job_level,
    em.job_role,
    em.business_travel,
    em.total_working_years,
    em.years_at_company,
    em.years_in_current_role,
    em.years_since_last_promotion,
    em.years_with_curr_manager,
    em.num_companies_worked,
    em.distance_from_home,
    em.overtime,

    c.daily_rate,
    c.hourly_rate,
    c.monthly_income,
    c.monthly_rate,
    c.percent_salary_hike,
    c.stock_option_level,

    en.environment_satisfaction,
    en.job_involvement,
    en.job_satisfaction,
    en.relationship_satisfaction,
    en.work_life_balance,

    p.performance_rating,
    p.training_times_last_year,

    a.attrition,
    a.attrition_flag

FROM employees e

LEFT JOIN employment em
    ON e.employee_id = em.employee_id

LEFT JOIN compensation c
    ON e.employee_id = c.employee_id

LEFT JOIN engagement en
    ON e.employee_id = en.employee_id

LEFT JOIN performance_learning p
    ON e.employee_id = p.employee_id

LEFT JOIN attrition a
    ON e.employee_id = a.employee_id;
"""

with engine.begin() as connection:
    connection.execute(text(create_view_query))

print("workforce_analytics view created successfully.")

workforce_analytics view created successfully.


In [66]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM workforce_analytics;
        """)
    )

    count = result.scalar()

print("Records in workforce_analytics:", count)

Records in workforce_analytics: 1470


In [67]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_name = 'workforce_analytics'
            ORDER BY ordinal_position;
        """)
    )

    columns = result.fetchall()

print("workforce_analytics columns:")

for column in columns:
    print(column[0])

workforce_analytics columns:
employee_id
age
gender
marital_status
education
education_field
department
job_level
job_role
business_travel
total_working_years
years_at_company
years_in_current_role
years_since_last_promotion
years_with_curr_manager
num_companies_worked
distance_from_home
overtime
daily_rate
hourly_rate
monthly_income
monthly_rate
percent_salary_hike
stock_option_level
environment_satisfaction
job_involvement
job_satisfaction
relationship_satisfaction
work_life_balance
performance_rating
training_times_last_year
attrition
attrition_flag


In [68]:
query = """
SELECT
    department,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY department
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

for row in rows:
    print(row)

('Human Resources', 63, None, None)
('Research & Development', 961, None, None)
('Sales', 446, None, None)


In [69]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS total_records,
                COUNT(attrition_flag) AS non_null_flags,
                COUNT(*) - COUNT(attrition_flag) AS null_flags,
                SUM(attrition_flag) AS total_attrition
            FROM attrition;
        """)
    )

    row = result.fetchone()

print("Total attrition records:", row[0])
print("Non-null attrition flags:", row[1])
print("NULL attrition flags:", row[2])
print("Total employees who left:", row[3])

Total attrition records: 0
Non-null attrition flags: 0
NULL attrition flags: 0
Total employees who left: None


In [70]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT *
            FROM attrition
            LIMIT 10;
        """)
    )

    rows = result.fetchall()

for row in rows:
    print(row)

In [71]:
attrition_df = df[
    [
        "EmployeeID",
        "Attrition",
        "AttritionFlag"
    ]
].copy()

attrition_df = attrition_df.rename(
    columns={
        "EmployeeID": "employee_id",
        "Attrition": "attrition",
        "AttritionFlag": "attrition_flag"
    }
)

print(attrition_df.head())
print()
print("Rows:", len(attrition_df))
print("Unique Employee IDs:", attrition_df["employee_id"].nunique())
print("Attrition values:")
print(attrition_df["attrition"].value_counts())
print()
print("Attrition flag values:")
print(attrition_df["attrition_flag"].value_counts())

   employee_id attrition  attrition_flag
0            1       Yes               1
1            2        No               0
2            4       Yes               1
3            5        No               0
4            7        No               0

Rows: 1470
Unique Employee IDs: 1470
Attrition values:
attrition
No     1233
Yes     237
Name: count, dtype: int64

Attrition flag values:
attrition_flag
0    1233
1     237
Name: count, dtype: int64


In [72]:
attrition_df.to_sql(
    "attrition",
    engine,
    if_exists="append",
    index=False
)

print("Attrition data loaded successfully.")

Attrition data loaded successfully.


In [73]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS total_records,
                COUNT(DISTINCT employee_id) AS unique_employees,
                SUM(attrition_flag) AS employees_left
            FROM attrition;
        """)
    )

    row = result.fetchone()

print("Total attrition records:", row[0])
print("Unique employees:", row[1])
print("Employees who left:", row[2])

Total attrition records: 1470
Unique employees: 1470
Employees who left: 237


In [74]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM attrition a
            LEFT JOIN employees e
                ON a.employee_id = e.employee_id
            WHERE e.employee_id IS NULL;
        """)
    )

    unmatched = result.scalar()

print("Unmatched attrition records:", unmatched)

Unmatched attrition records: 0


In [75]:
query = """
SELECT
    department,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY department
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

for row in rows:
    print(row)

('Sales', 446, 92, Decimal('20.63'))
('Human Resources', 63, 12, Decimal('19.05'))
('Research & Development', 961, 133, Decimal('13.84'))


In [76]:
query = """
SELECT
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    COUNT(*) - SUM(attrition_flag) AS employees_stayed,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate,
    ROUND(AVG(age), 2) AS average_age,
    ROUND(AVG(monthly_income), 2) AS average_monthly_income,
    ROUND(AVG(job_satisfaction), 2) AS average_job_satisfaction,
    ROUND(AVG(work_life_balance), 2) AS average_work_life_balance
FROM workforce_analytics;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    row = result.fetchone()

print("===== WORKFORCE KPI SUMMARY =====")
print("Total Employees:", row[0])
print("Employees Left:", row[1])
print("Employees Stayed:", row[2])
print("Attrition Rate:", row[3], "%")
print("Average Age:", row[4])
print("Average Monthly Income:", row[5])
print("Average Job Satisfaction:", row[6])
print("Average Work-Life Balance:", row[7])

===== WORKFORCE KPI SUMMARY =====
Total Employees: 1470
Employees Left: 237
Employees Stayed: 1233
Attrition Rate: 16.12 %
Average Age: 36.92
Average Monthly Income: 6502.93
Average Job Satisfaction: 2.73
Average Work-Life Balance: 2.76


In [77]:
query = """
SELECT
    department,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY department
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY DEPARTMENT =====")

for row in rows:
    print(
        f"Department: {row[0]} | "
        f"Employees: {row[1]} | "
        f"Left: {row[2]} | "
        f"Attrition Rate: {row[3]}%"
    )

===== ATTRITION BY DEPARTMENT =====
Department: Sales | Employees: 446 | Left: 92 | Attrition Rate: 20.63%
Department: Human Resources | Employees: 63 | Left: 12 | Attrition Rate: 19.05%
Department: Research & Development | Employees: 961 | Left: 133 | Attrition Rate: 13.84%


In [78]:
query = """
SELECT
    overtime,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY overtime
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY OVERTIME =====")

for row in rows:
    print(row)

===== ATTRITION BY OVERTIME =====
('Yes', 416, 127, Decimal('30.53'))
('No', 1054, 110, Decimal('10.44'))


In [79]:
query = """
SELECT
    job_satisfaction,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY job_satisfaction
ORDER BY job_satisfaction;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY JOB SATISFACTION =====")

for row in rows:
    print(row)

===== ATTRITION BY JOB SATISFACTION =====
(1, 289, 66, Decimal('22.84'))
(2, 280, 46, Decimal('16.43'))
(3, 442, 73, Decimal('16.52'))
(4, 459, 52, Decimal('11.33'))


In [80]:
query = """
SELECT
    work_life_balance,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY work_life_balance
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY WORK-LIFE BALANCE =====")

for row in rows:
    print(row)

===== ATTRITION BY WORK-LIFE BALANCE =====
(1, 80, 25, Decimal('31.25'))
(4, 153, 27, Decimal('17.65'))
(2, 344, 58, Decimal('16.86'))
(3, 893, 127, Decimal('14.22'))


In [81]:
query = """
SELECT
    CASE
        WHEN monthly_income < 3000 THEN 'Low Income'
        WHEN monthly_income BETWEEN 3000 AND 6000 THEN 'Medium Income'
        ELSE 'High Income'
    END AS income_group,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY income_group
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY INCOME GROUP =====")

for row in rows:
    print(row)

===== ATTRITION BY INCOME GROUP =====
('Low Income', 395, 113, Decimal('28.61'))
('Medium Income', 519, 66, Decimal('12.72'))
('High Income', 556, 58, Decimal('10.43'))


In [82]:
query = """
SELECT
    CASE
        WHEN years_at_company <= 2 THEN '0-2 Years'
        WHEN years_at_company <= 5 THEN '3-5 Years'
        WHEN years_at_company <= 10 THEN '6-10 Years'
        ELSE '10+ Years'
    END AS tenure_group,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY tenure_group
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY TENURE =====")

for row in rows:
    print(row)

===== ATTRITION BY TENURE =====
('0-2 Years', 342, 102, Decimal('29.82'))
('3-5 Years', 434, 60, Decimal('13.82'))
('6-10 Years', 448, 55, Decimal('12.28'))
('10+ Years', 246, 20, Decimal('8.13'))


In [84]:
query = """
SELECT
    CASE
        WHEN age < 30 THEN 'Under 30'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        ELSE '50+'
    END AS age_group,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY
    CASE
        WHEN age < 30 THEN 'Under 30'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        ELSE '50+'
    END
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY AGE GROUP =====")

for row in rows:
    print(row)

===== ATTRITION BY AGE GROUP =====
('Under 30', 326, 91, Decimal('27.91'))
('30-39', 622, 89, Decimal('14.31'))
('50+', 173, 23, Decimal('13.29'))
('40-49', 349, 34, Decimal('9.74'))


In [85]:
query = """
SELECT
    job_involvement,
    COUNT(*) AS total_employees,
    SUM(attrition_flag) AS employees_left,
    ROUND(
        100.0 * SUM(attrition_flag) / COUNT(*),
        2
    ) AS attrition_rate
FROM workforce_analytics
GROUP BY job_involvement
ORDER BY attrition_rate DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    rows = result.fetchall()

print("===== ATTRITION BY JOB INVOLVEMENT =====")

for row in rows:
    print(row)

===== ATTRITION BY JOB INVOLVEMENT =====
(1, 83, 28, Decimal('33.73'))
(2, 375, 71, Decimal('18.93'))
(3, 868, 125, Decimal('14.40'))
(4, 144, 13, Decimal('9.03'))


In [86]:
query = """
SELECT *
FROM workforce_analytics;
"""

with engine.connect() as connection:
    workforce_df = pd.read_sql(text(query), connection)

print("Dataset loaded from PostgreSQL.")
print("Shape:", workforce_df.shape)
print("\nColumns:")
print(workforce_df.columns.tolist())

Dataset loaded from PostgreSQL.
Shape: (1470, 33)

Columns:
['employee_id', 'age', 'gender', 'marital_status', 'education', 'education_field', 'department', 'job_level', 'job_role', 'business_travel', 'total_working_years', 'years_at_company', 'years_in_current_role', 'years_since_last_promotion', 'years_with_curr_manager', 'num_companies_worked', 'distance_from_home', 'overtime', 'daily_rate', 'hourly_rate', 'monthly_income', 'monthly_rate', 'percent_salary_hike', 'stock_option_level', 'environment_satisfaction', 'job_involvement', 'job_satisfaction', 'relationship_satisfaction', 'work_life_balance', 'performance_rating', 'training_times_last_year', 'attrition', 'attrition_flag']


In [87]:
print("First 5 rows:")
display(workforce_df.head())

print("\nDataset information:")
print(workforce_df.info())

First 5 rows:


,employee_id,age,gender,marital_status,education,education_field,department,job_level,job_role,business_travel,...,stock_option_level,environment_satisfaction,job_involvement,job_satisfaction,relationship_satisfaction,work_life_balance,performance_rating,training_times_last_year,attrition,attrition_flag
0,1,41,Female,Single,2,Life Sciences,Sales,2,Sales Executive,Travel_Rarely,...,0,2,3,4,1,1,3,0,Yes,1
1,2,49,Male,Married,1,Life Sciences,Research & Development,2,Research Scientist,Travel_Frequently,...,1,3,2,2,4,3,4,3,No,0
2,4,37,Male,Single,2,Other,Research & Development,1,Laboratory Technician,Travel_Rarely,...,0,4,2,3,2,3,3,3,Yes,1
3,5,33,Female,Married,4,Life Sciences,Research & Development,1,Research Scientist,Travel_Frequently,...,0,4,3,3,3,3,3,3,No,0
4,7,27,Male,Married,1,Medical,Research & Development,1,Laboratory Technician,Travel_Rarely,...,1,1,3,2,4,3,3,3,No,0



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   employee_id                 1470 non-null   int64 
 1   age                         1470 non-null   int64 
 2   gender                      1470 non-null   object
 3   marital_status              1470 non-null   object
 4   education                   1470 non-null   int64 
 5   education_field             1470 non-null   object
 6   department                  1470 non-null   object
 7   job_level                   1470 non-null   int64 
 8   job_role                    1470 non-null   object
 9   business_travel             1470 non-null   object
 10  total_working_years         1470 non-null   int64 
 11  years_at_company            1470 non-null   int64 
 12  years_in_current_role       1470 non-null   int64 
 13  years_since_last_promotion

In [88]:
print("Attrition distribution:")
print(workforce_df["attrition_flag"].value_counts())

print("\nAttrition percentage:")
print(
    workforce_df["attrition_flag"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Attrition distribution:
attrition_flag
0    1233
1     237
Name: count, dtype: int64

Attrition percentage:
attrition_flag
0    83.88
1    16.12
Name: proportion, dtype: float64


In [89]:
print("===== MISSING VALUES =====")

missing = workforce_df.isnull().sum()

print(missing[missing > 0])

===== MISSING VALUES =====
Series([], dtype: int64)


In [90]:
print("===== DUPLICATE ROWS =====")

duplicates = workforce_df.duplicated().sum()

print("Duplicate rows:", duplicates)

===== DUPLICATE ROWS =====
Duplicate rows: 0


In [91]:
print("===== UNIQUE VALUES =====")

for column in workforce_df.columns:
    print(
        f"{column}: "
        f"{workforce_df[column].nunique()} unique values"
    )

===== UNIQUE VALUES =====
employee_id: 1470 unique values
age: 43 unique values
gender: 2 unique values
marital_status: 3 unique values
education: 5 unique values
education_field: 6 unique values
department: 3 unique values
job_level: 5 unique values
job_role: 9 unique values
business_travel: 3 unique values
total_working_years: 40 unique values
years_at_company: 37 unique values
years_in_current_role: 19 unique values
years_since_last_promotion: 16 unique values
years_with_curr_manager: 18 unique values
num_companies_worked: 10 unique values
distance_from_home: 29 unique values
overtime: 2 unique values
daily_rate: 886 unique values
hourly_rate: 71 unique values
monthly_income: 1349 unique values
monthly_rate: 1427 unique values
percent_salary_hike: 15 unique values
stock_option_level: 4 unique values
environment_satisfaction: 4 unique values
job_involvement: 4 unique values
job_satisfaction: 4 unique values
relationship_satisfaction: 4 unique values
work_life_balance: 4 unique values